In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("Gaming_Academic_Performance.csv")

In [ ]:
print(df.head())

In [ ]:
print(df.info())

In [ ]:
print(df.describe())

In [ ]:
print(len(df.columns))

we have 8k records with 14 features. Need to check target variable distribution

In [ ]:
y = df["gender"]
print(y.value_counts())

right here we are interested in binary classification, so lets use knn for replace 3rd variable

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [ ]:
train_df = df[df['gender'].isin(['Male', 'Female'])]
unknown_df = df[df['gender'] == 'Other'].copy()

# (Male=0, Female=1)
le = LabelEncoder()
y_train = le.fit_transform(train_df['gender'])

In [ ]:
feature_cols = df.select_dtypes(include=['int64', 'float64']).columns
X_train = train_df[feature_cols].copy()
X_unknown = unknown_df[feature_cols].copy()

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_unknown_scaled = scaler.transform(X_unknown)

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

y_pred = knn.predict(X_unknown_scaled)
predicted_genders = le.inverse_transform(y_pred)
df.loc[df['gender'] == 'Other', 'gender'] = predicted_genders

In [ ]:
print(df['gender'].value_counts())

now we have binary task, we can try some models

In [ ]:
#base model
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.preprocessing import  StandardScaler

In [ ]:
X = df.select_dtypes(include=['int64', 'float64'])
y = df["gender"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
base_model = LogisticRegression(max_iter=100)
base_model.fit(X_train, y_train)
y_pred = base_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
martix = confusion_matrix(y_test, y_pred)

In [ ]:
print(accuracy)

we will use accuracy as marker

In [ ]:
correlation_matrix = X.corr()

high_corr = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        if abs(correlation_matrix.iloc[i, j]) > 0.8:
            high_corr.append((correlation_matrix.columns[i],
                            correlation_matrix.columns[j],
                            correlation_matrix.iloc[i, j]))

print("Highly correlated pairs (>0.8):")
for pair in high_corr:
    print(f"  {pair[0]} - {pair[1]}: {pair[2]:.3f}")

In [125]:
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

cols_to_remove = ['student_id', 'device_usage', 'reaction_time_ms', 'gaming_hours']

X_1 = df.select_dtypes(include=['int64', 'float64']).drop(columns=cols_to_remove)
y = df["gender"]
X_train_1, X_test_1, y_train_1, y_test_1 = train_test_split(X_1, y, test_size=0.2, random_state=42)

In [120]:
model_no_corr = LogisticRegression(max_iter=1000)
model_no_corr.fit(X_train_1, y_train_1)
y_pred_1 = model_no_corr.predict(X_test_1)
ac_1 = accuracy_score(y_test_1, y_pred_1)

In [121]:
print(ac_1)

0.495625


In [122]:
from sklearn.tree import DecisionTreeClassifier

In [123]:
d_t = DecisionTreeClassifier()
d_t.fit(X_train_1, y_train_1)
y_pred_2 = d_t.predict(X_test_1)
ac_2 = accuracy_score(y_test_1, y_pred_2)

In [124]:
print(ac_2)

0.513125


In [126]:
from sklearn.ensemble import RandomForestClassifier

In [131]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_1, y_train_1)
y_pred_3 = rf.predict(X_test_1)
ac_3 = accuracy_score(y_test_1, y_pred_3)

In [132]:
print(ac_3)

0.489375


through all this attempts we can conclude that we cant know gender from other features in this data